# 42_02 - Model serangan: demo & verifikasi (Paper 4-2, Fase 1)

**Tujuan:** verifikasi tiga serangan dari `robust_agg.py` BENAR-BENAR merusak
model (sebelum menguji pertahanan). Tiga serangan: (1) *label-flipping*,
(2) penskalaan bobot, (3) *backdoor* (trigger -> target benign).

**Sumber tunggal:** semua logika serangan + agregasi dari `robust_agg.py`
(di-reuse semua nb 42). MLP/forward/train dari `../aws/fed_node.py`.

**Output:** `fed_out42/attack_demo.csv` (efek tiap serangan vs baseline bersih).

> Jalankan setelah 42_01 (butuh `fed_out42/parts_K10.npz`). JANGAN mengarang angka.

## 0. Setup: impor fed_node + robust_agg

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.'))
import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEED=42
print('fed LAYERS', fed.LAYERS, '| robust_agg rules siap')

## 1. Muat partisi K-klien (dari 42_01) + global test

In [ ]:
def need(path):
    assert os.path.exists(path), 'PRASYARAT GAGAL: %s tak ada - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')), allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')), allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
print('K',K,'| pool',X.shape,'| global-test',Xg.shape)

## 2. Fungsi FedAvg satu-putaran lengkap (reuse fed_node + robust_agg)

Tiap klien: z-score lokal, latih `train_local`. Server: agregasi via `ra.aggregate`.
Serangan disuntikkan di level klien (data) atau update (bobot) sesuai jenis.

In [ ]:
def predict_with(W, mu, sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5,
            rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    # pra-hitung data & z-score lokal tiap klien
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy()
        rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        mu,sd=fed.zscore_fit(Xk) if len(yk) else (np.zeros(X.shape[1]),np.ones(X.shape[1]))
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd), y=yk, n=len(yk)))
    Wg=fed.init_weights(seed)
    f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(rounds):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg, cli[k]['X'], cli[k]['y'], epochs=epochs, lr=lr, mu=0.0, W_global=Wg, seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl, gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule, wl, sizes, f=f_byz, beta=max(1,f_byz))
    # evaluasi pakai z-score GLOBAL train (monitoring konsisten)
    mu_g,sd_g=fed.zscore_fit(X)
    pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    mcc=round(matthews_corrcoef(yg,yp),4)
    asr=round(ra.attack_success_rate(pf, Xg, yg),4) if attack=='backdoor' else None
    return mcc, asr, Wg
print('run_fed siap')

## 3. Demo: baseline bersih vs tiap serangan (FedAvg, f=20% -> 2 klien jahat)

Tujuan verifikasi: tiap serangan HARUS menurunkan MCC (atau ASR tinggi utk backdoor)
dibanding baseline bersih. Jika TIDAK, serangan tak efektif -> perlu kalibrasi
(dicatat jujur, bukan dipaksakan).

In [ ]:
mal2=(0,1)   # 2 klien jahat (f=20% dari K=10)
rows=[]
mcc0,_,_=run_fed('fedavg', attack=None)
rows.append({'kondisi':'bersih (tanpa serangan)','rule':'fedavg','MCC':mcc0,'ASR':None})
for atk in ['label_flip','scale','backdoor']:
    mcc,asr,_=run_fed('fedavg', attack=atk, mal_ids=mal2)
    rows.append({'kondisi':'serangan '+atk+' (f=20%)','rule':'fedavg','MCC':mcc,'ASR':asr})
    print(atk,'-> MCC',mcc,'ASR',asr)
dfd=pd.DataFrame(rows); dfd.to_csv(os.path.join(OUTDIR,'attack_demo.csv'),index=False)
import IPython.display as ipd; ipd.display(dfd)
print('CEK: MCC turun (label_flip/scale) & ASR tinggi (backdoor) = serangan efektif.')
print('=== SEL 3 (demo serangan) SELESAI ===')

## 4. Catatan untuk naskah

- Verifikasi bahwa serangan EFEKTIF adalah prasyarat sebelum menguji pertahanan
  (42_05/42_06). Jika suatu serangan lemah pada ruang SFM, dilaporkan apa adanya.
- `attack_demo.csv` mendukung narasi RQ1 (kerentanan FedAvg) di `paper-4-2.tex`.
- Backdoor: trigger = fitur [0, 1, 2] di-set 5.0; target label benign(0). ASR tinggi =
  serangan lolos deteksi. (parameter trigger terpusat di `robust_agg.py`).
- Semua logika dari `robust_agg.py` + `fed_node.py` -> konsisten dgn nb lain & EC2.